#  Flight Delay Prediction
## Phase 3 — Exploratory Data Analysis (EDA)

---

| Field | Details |
|---|---|
| **Course** | COMP4381 – Data Science and Analytics, Spring 2026 |
| **Instructor** | Ahmed Sabbah |



---

## 1. Introduction

This notebook provides a comprehensive Exploratory Data Analysis (EDA) of a massive dataset on flight delays. This analysis aims to:


- Understand the structure and quality of the raw data
- Identify missing, duplicate, and anomalous values
- Summarize the key statistical characteristics of the delay-related variables
- Extract insights to guide feature engineering and modeling

>**Dataset Size:** 1,747,627 flight records across 9 major US airports

---
## 2. Dataset Source

| Property | Value |
|---|---|
| **Source** | Kaggle – Flight Delay Dataset |
| **Total Records** | 1,747,627 flights |
| **Total Features** | 16 columns |
| **Airports Covered** | ATL, DFW, JFK, LAX, ORD, BOS, MIA, SEA, SFO |
| **File Path** | `../data/raw/flight_delays.csv` |

The dataset contains scheduled and actual departure/arrival times, airline codes, departure and arrival airports, and delay information in minutes.


---
## 3. Import Libraries

We use the following standard Python libraries:

- **`pandas`** — data loading, cleaning, and manipulation
- **`numpy`** — numerical operations and array handling
- **`matplotlib` / `seaborn`** — visualisation


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('[OK] Libraries imported successfully!')

[OK] Libraries imported successfully!


---
## 4. Load Dataset

The raw dataset is stored in the path `data/raw/flight_delays.csv` according to the project folder structure. We load it using the function `pandas.read_csv()` and immediately check its format.


In [4]:
df = pd.read_csv('../data/raw/flight_delays.csv')
print(f'[OK] Data loaded: {df.shape[0]:,} rows, {df.shape[1]:,} columns')

[OK] Data loaded: 1,747,627 rows, 16 columns


---
## 5. Dataset Structure

We review the first few rows to understand the data structure before performing any transformations. This helps in detecting obvious formatting issues or unexpected values ​​early on.


In [5]:
print('=' * 70)
print('DATASET STRUCTURE — FIRST 5 ROWS')
print('=' * 70)
df.head()

DATASET STRUCTURE — FIRST 5 ROWS


,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance
0,1,United,4558,ORD,MIA,2024-09-01 08:11,2024-09-01 08:30,2024-09-01 12:11,2024-09-01 12:19,8,Weather,True,False,Boeing 737,N71066,1031
1,2,Delta,8021,LAX,MIA,2024-09-01 10:25,2024-09-01 10:41,2024-09-01 13:25,2024-09-01 13:27,2,Air Traffic Control,True,True,Airbus A320,N22657,1006
2,3,Southwest,7520,DFW,SFO,2024-09-01 16:53,2024-09-01 17:05,2024-09-01 17:53,2024-09-01 18:07,14,Weather,True,True,Boeing 737,N95611,2980
3,4,Delta,2046,ORD,BOS,2024-09-01 14:44,2024-09-01 15:04,2024-09-01 18:44,2024-09-01 18:34,-10,NaN,False,False,Boeing 777,N90029,1408
4,5,Delta,6049,LAX,SEA,2024-09-01 01:51,2024-09-01 02:08,2024-09-01 05:51,2024-09-01 06:15,24,Air Traffic Control,False,True,Boeing 737,N27417,2298


---
## 6. Dataset Dimensions

We confirm the total number of rows and columns to verify the data loaded completely.

In [6]:
print('=' * 70)
print('DATASET DIMENSIONS')
print('=' * 70)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]:,}')

DATASET DIMENSIONS
  Rows    : 1,747,627
  Columns : 16


---
## 7. Column Names & Types

Understanding the data type of each column is essential before any cleanup step. Mismatched types (such as a numeric column stored as `object`) are a common source of errors.


In [7]:
print('=' * 70)
print('COLUMN NAMES')
print('=' * 70)
for idx, col in enumerate(df.columns, 1):
    print(f'  {idx:2d}. {col}')

COLUMN NAMES
   1. FlightID
   2. Airline
   3. FlightNumber
   4. Origin
   5. Destination
   6. ScheduledDeparture
   7. ActualDeparture
   8. ScheduledArrival
   9. ActualArrival
  10. DelayMinutes
  11. DelayReason
  12. Cancelled
  13. Diverted
  14. AircraftType
  15. TailNumber
  16. Distance


In [8]:
print('=' * 70)
print('DATA TYPE DISTRIBUTION')
print('=' * 70)
for dtype, count in df.dtypes.value_counts().items():
    print(f'  {str(dtype):20s}: {count} column(s)')

DATA TYPE DISTRIBUTION
  object              : 10 column(s)
  int64               : 4 column(s)
  bool                : 2 column(s)


In [9]:
print('=' * 70)
print('COLUMN SUMMARY TABLE')
print('=' * 70)
desc = pd.DataFrame({
    'Column'  : df.columns,
    'Type'    : df.dtypes.values,
    'Missing' : df.isnull().sum().values
})
print(desc.to_string(index=False))

COLUMN SUMMARY TABLE
            Column   Type  Missing
          FlightID  int64        0
           Airline object        0
      FlightNumber  int64        0
            Origin object        0
       Destination object        0
ScheduledDeparture object        0
   ActualDeparture object        0
  ScheduledArrival object        0
     ActualArrival object        0
      DelayMinutes  int64        0
       DelayReason object   468873
         Cancelled   bool        0
          Diverted   bool        0
      AircraftType object        0
        TailNumber object        0
          Distance  int64        0


---
## 8. Missing Values Analysis

Incomplete data can lead to model bias or failure. Here, we calculate the overall data completeness as a percentage and indicate any columns that require missing data to be completed or deleted.

A data completeness rate exceeding 95% is generally considered acceptable for tabular datasets.

In [10]:
completeness = (1 - (df.isnull().sum().sum() / (len(df) * len(df.columns)))) * 100
print(f'  Data Completeness : {completeness:.2f}%')

missing_per_col = df.isnull().sum()
missing_cols = missing_per_col[missing_per_col > 0]
if missing_cols.empty:
    print('  [OK] No missing values detected in any column.')
else:
    print('\n  Columns with missing values:')
    print(missing_cols.to_string())

  Data Completeness : 98.32%

  Columns with missing values:
DelayReason    468873


---
## 9. Duplicate Records

Duplicate rows may provide a distortion of the statistical and training summaries. We will inquire again to confirm the presence of duplicate questions and report the number.


In [11]:
dups = df.duplicated().sum()
print(f'  Duplicate rows : {dups:,}')
if dups == 0:
    print('  [OK] No duplicate records found — data is unique.')
else:
    print(f'  [!] {dups:,} duplicate rows detected. Consider removing them before modelling.')

  Duplicate rows : 0
  [OK] No duplicate records found — data is unique.


---
## 10. Descriptive Statistics

We calculate standard summary statistics (count, mean, standard deviation, minimum, quartiles, maximum) for all **numeric columns**. This helps determine the range and spread of delay values ​​and other continuous properties.

In [12]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe()

,FlightID,FlightNumber,DelayMinutes,Distance
count,1747627.00,1747627.00,1747627.00,1747627.00
mean,873814.00,5001.16,10.00,1549.94
std,504496.60,2885.82,11.83,836.87
min,1.00,1.00,-10.00,100.00
25%,436907.50,2503.00,0.00,825.00
50%,873814.00,5002.00,10.00,1551.00
75%,1310720.50,7499.00,20.00,2274.00
max,1747627.00,9999.00,30.00,3000.00


---
## 11. Delay Analysis

The key variable of interest is "delay duration in minutes." Here we will examine the following:

- **Average delay** for all flights
- **Number of delayed flights** (where "delay duration in minutes" > 0)
- **Percentage of delayed flights** to the total number of flights

This provides an initial picture of the frequency and severity of delays.

In [13]:
delay_stats = df['DelayMinutes'].describe()
delayed     = (df['DelayMinutes'] > 0).sum()
delay_pct   = (delayed / len(df)) * 100

print('=' * 70)
print('DELAY SUMMARY')
print('=' * 70)
print(f'  Mean Delay       : {delay_stats["mean"]:.2f} minutes')
print(f'  Median Delay     : {delay_stats["50%"]:.2f} minutes')
print(f'  Max Delay        : {delay_stats["max"]:.2f} minutes')
print(f'  Delayed Flights  : {delayed:,}  ({delay_pct:.2f}% of total)')

DELAY SUMMARY
  Mean Delay       : 10.00 minutes
  Median Delay     : 10.00 minutes
  Max Delay        : 30.00 minutes
  Delayed Flights  : 1,278,754  (73.17% of total)


---
## 12. Outlier Detection

We use the **Interquartile Range (IQR) method** to detect outliers in `DelayMinutes`. A value is flagged as an outlier if it falls below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.

Extreme delay values may represent genuine disruptions (e.g. severe weather, mechanical failure) or data entry errors — both require careful treatment in the modelling phase.

In [14]:
Q1  = df['DelayMinutes'].quantile(0.25)
Q3  = df['DelayMinutes'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ((df['DelayMinutes'] < lower_bound) | (df['DelayMinutes'] > upper_bound)).sum()

print('=' * 70)
print('OUTLIER DETECTION — DelayMinutes (IQR Method)')
print('=' * 70)
print(f'  Q1 (25th pct)    : {Q1:.2f} min')
print(f'  Q3 (75th pct)    : {Q3:.2f} min')
print(f'  IQR              : {IQR:.2f} min')
print(f'  Lower Bound      : {lower_bound:.2f} min')
print(f'  Upper Bound      : {upper_bound:.2f} min')
print(f'  Outliers Found   : {outliers:,}')

OUTLIER DETECTION — DelayMinutes (IQR Method)
  Q1 (25th pct)    : 0.00 min
  Q3 (75th pct)    : 20.00 min
  IQR              : 20.00 min
  Lower Bound      : -30.00 min
  Upper Bound      : 50.00 min
  Outliers Found   : 0


---
## 13. Airport Analysis

All historical airports that were either **level airports** or **destination airports** were identified in the dataset.

In [15]:
origins      = df['Origin'].unique()
destinations = df['Destination'].unique()
all_airports = sorted(set(list(origins) + list(destinations)))

print('=' * 70)
print('AIRPORT COVERAGE')
print('=' * 70)
print(f'  Origin airports      : {len(origins)}')
print(f'  Destination airports : {len(destinations)}')
print(f'  Total unique airports: {len(all_airports)}')
print(f'  Airport codes        : {", ".join(all_airports)}')

AIRPORT COVERAGE
  Origin airports      : 5
  Destination airports : 5
  Total unique airports: 9
  Airport codes        : ATL, BOS, DFW, JFK, LAX, MIA, ORD, SEA, SFO


---
## 14. Airline Distribution

We examine how flights are distributed among airlines. A highly unbalanced distribution (the dominance of one airline) can lead to bias in the model, so this step is useful in any subsequent decisions regarding stratified sampling.

In [31]:
airline_counts = df['Airline'].value_counts()

print('=' * 70)
print('AIRLINE DISTRIBUTION')
print('=' * 70)
for airline, count in airline_counts.items():
    pct = (count / len(df)) * 100
    bar = '' * int(pct / 2)
    print(f'  {airline:6s} : {count:>9,}  ({pct:5.2f}%)  {bar}')

AIRLINE DISTRIBUTION
  Southwest :   437,721  (25.05%)  
  American Airlines :   437,124  (25.01%)  
  Delta  :   436,680  (24.99%)  
  United :   436,102  (24.95%)  


# **15. Data Cleaning** 
 This section focuses on improving the quality of the dataset by identifying and removing duplicate records. Duplicate entries may lead to inaccurate analytical results and affect future machine learning models.   



In [17]:
print("="*70)
print("DATA CLEANING")
print("="*70)

df_clean = df.copy()

print(f"Original Dataset Shape: {df_clean.shape}")

duplicates = df_clean.duplicated().sum()

print(f"\nNumber of Duplicate Records: {duplicates}")

df_clean = df_clean.drop_duplicates()

print(f"Dataset Shape After Removing Duplicates: {df_clean.shape}")

DATA CLEANING
Original Dataset Shape: (1747627, 16)

Number of Duplicate Records: 0
Dataset Shape After Removing Duplicates: (1747627, 16)


# **16. Missing Values Analysis**

Missing values can negatively affect statistical analysis and predictive modeling. This step identifies missing values across all dataset attributes.

In [18]:
print("="*70)
print("MISSING VALUES ANALYSIS")
print("="*70)

missing_values = df_clean.isnull().sum()

print("Missing Values Per Column:")
print(missing_values[missing_values > 0])

MISSING VALUES ANALYSIS
Missing Values Per Column:
DelayReason    468873
dtype: int64


The output shows the number of missing values in each feature. Features containing missing values require appropriate treatment before further processing.

The output shows the number of missing values in each feature. Features containing missing values require appropriate treatment before further processing.

# **17. Missing Values Treatment**
Handle missing values to improve data completeness and maintain dataset quality.

In [19]:
before_rows = len(df_clean)

df_clean = df_clean.dropna()

after_rows = len(df_clean)

print(f"Rows Before Cleaning: {before_rows}")
print(f"Rows After Cleaning: {after_rows}")
print(f"Removed Rows: {before_rows - after_rows}")

Rows Before Cleaning: 1747627
Rows After Cleaning: 1278754
Removed Rows: 468873


Rows containing missing values were removed from the dataset. This ensures that all remaining observations contain complete information.

# **18. Data Quality Verification**
    
Verify that all duplicates and missing values have been successfully handled.

In [20]:
print("="*70)
print("DATA QUALITY VERIFICATION")
print("="*70)

print("Remaining Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nRemaining Duplicate Records:")
print(df_clean.duplicated().sum())

DATA QUALITY VERIFICATION
Remaining Missing Values:
0

Remaining Duplicate Records:
0


The dataset was validated after cleaning to ensure that no duplicate records or missing values remain.

# **19. Data Preprocessing**

Prepare the dataset for feature engineering by standardizing column names and converting date/time columns into appropriate formats.

In [21]:
print("="*70)
print("DATA PREPROCESSING")
print("="*70)

df_clean.columns = df_clean.columns.str.strip()

date_columns = [
    "ScheduledDeparture",
    "ActualDeparture",
    "ScheduledArrival",
    "ActualArrival"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col])

print("Date and Time Conversion Completed Successfully")

DATA PREPROCESSING
Date and Time Conversion Completed Successfully


The date and time columns were converted into datetime format, allowing time-based calculations and feature extraction.

# **20. Feature Engineering: Departure Delay**

Calculate the difference between actual and scheduled departure times.

In [22]:
df_clean["DepartureDelay"] = (
    df_clean["ActualDeparture"] -
    df_clean["ScheduledDeparture"]
).dt.total_seconds() / 60

df_clean["DepartureDelay"].describe()

count   1278754.00
mean         15.00
std           8.95
min           0.00
25%           7.00
50%          15.00
75%          23.00
max          30.00
Name: DepartureDelay, dtype: float64

DepartureDelay measures the number of minutes a flight departed earlier or later than scheduled.

# **21. Feature Engineering: Arrival Delay**

Calculate the difference between actual and scheduled arrival times.

In [23]:
df_clean["ArrivalDelay"] = (
    df_clean["ActualArrival"] -
    df_clean["ScheduledArrival"]
).dt.total_seconds() / 60

df_clean["ArrivalDelay"].describe()

count   1278754.00
mean         15.50
std           8.66
min           1.00
25%           8.00
50%          15.00
75%          23.00
max          30.00
Name: ArrivalDelay, dtype: float64

ArrivalDelay measures the difference between actual and planned arrival times and provides a useful indicator of operational performance.

# **22. Feature Engineering: Flight Duration**

Calculate the actual flight duration in minutes.

In [24]:
df_clean["FlightDuration"] = (
    df_clean["ActualArrival"] -
    df_clean["ActualDeparture"]
).dt.total_seconds() / 60

df_clean["FlightDuration"].describe()

count   1278754.00
mean        210.51
std         103.20
min          31.00
25%         120.00
50%         210.00
75%         301.00
max         390.00
Name: FlightDuration, dtype: float64

FlightDuration represents the actual travel time of each flight and may help explain delay patterns.

# **23. Creating Distance Category**

Group flights into distance categories for easier analysis.

In [25]:
def distance_category(distance):

    if distance < 1000:
        return "Short"

    elif distance < 2000:
        return "Medium"

    else:
        return "Long"

df_clean["DistanceCategory"] = df_clean["Distance"].apply(
    distance_category
)

df_clean["DistanceCategory"].value_counts()

DistanceCategory
Medium    441584
Long      440844
Short     396326
Name: count, dtype: int64

Flights were categorized into Short, Medium, and Long routes based on travel distance.

# **24. Creating Delay Category**

Classify flights into delayed and on-time groups.

In [26]:
df_clean["DelayCategory"] = df_clean["DelayMinutes"].apply(
    lambda x: "Delayed" if x > 15 else "On Time"
)

df_clean["DelayCategory"].value_counts()

DelayCategory
On Time    639437
Delayed    639317
Name: count, dtype: int64

Flights with delays greater than 15 minutes were classified as delayed, while all others were considered on time.

# **25. Encoding Categorical Variables**

Convert categorical variables into numerical values suitable for machine learning algorithms.

In [27]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

categorical_columns = [
    "Airline",
    "Origin",
    "Destination",
    "DelayReason",
    "AircraftType",
    "DistanceCategory",
    "DelayCategory"
]

for col in categorical_columns:

    if col in df_clean.columns:

        df_clean[col] = encoder.fit_transform(
            df_clean[col].astype(str)
        )

print("Encoding Completed Successfully")

Encoding Completed Successfully


All selected categorical features were encoded into numerical values, making the dataset compatible with machine learning techniques.

# **26. Final Dataset Validation**

Review the final dataset structure after cleaning and preprocessing.


In [28]:
print("="*70)
print("FINAL DATASET VALIDATION")
print("="*70)

print("Dataset Shape:")
print(df_clean.shape)

print("\nDataset Information:")
df_clean.info()

FINAL DATASET VALIDATION
Dataset Shape:
(1278754, 21)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
Index: 1278754 entries, 0 to 1747626
Data columns (total 21 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   FlightID            1278754 non-null  int64         
 1   Airline             1278754 non-null  int64         
 2   FlightNumber        1278754 non-null  int64         
 3   Origin              1278754 non-null  int64         
 4   Destination         1278754 non-null  int64         
 5   ScheduledDeparture  1278754 non-null  datetime64[ns]
 6   ActualDeparture     1278754 non-null  datetime64[ns]
 7   ScheduledArrival    1278754 non-null  datetime64[ns]
 8   ActualArrival       1278754 non-null  datetime64[ns]
 9   DelayMinutes        1278754 non-null  int64         
 10  DelayReason         1278754 non-null  int64         
 11  Cancelled           1278754 non-null  bool          
 12 

The final dataset structure was reviewed to ensure that all transformations were applied successfully.

# **27. Dataset Preview**

Display a sample of the final processed dataset.

In [29]:
df_clean.head()

,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance,DepartureDelay,ArrivalDelay,FlightDuration,DistanceCategory,DelayCategory
0,1,3,4558,4,2,2024-09-01 08:11:00,2024-09-01 08:30:00,2024-09-01 12:11:00,2024-09-01 12:19:00,8,2,True,False,1,N71066,1031,19.00,8.00,229.00,1,1
1,2,1,8021,3,2,2024-09-01 10:25:00,2024-09-01 10:41:00,2024-09-01 13:25:00,2024-09-01 13:27:00,2,0,True,True,0,N22657,1006,16.00,2.00,166.00,1,1
2,3,2,7520,1,4,2024-09-01 16:53:00,2024-09-01 17:05:00,2024-09-01 17:53:00,2024-09-01 18:07:00,14,2,True,True,1,N95611,2980,12.00,14.00,62.00,0,1
4,5,1,6049,3,3,2024-09-01 01:51:00,2024-09-01 02:08:00,2024-09-01 05:51:00,2024-09-01 06:15:00,24,0,False,True,1,N27417,2298,17.00,24.00,247.00,0,0
6,7,2,4188,4,1,2024-09-01 05:47:00,2024-09-01 06:02:00,2024-09-01 10:47:00,2024-09-01 10:54:00,7,2,False,True,2,N25382,1674,15.00,7.00,292.00,1,1


The preview provides a quick overview of the cleaned and transformed dataset.

# **24. Saving the Processed Dataset**

Save the processed dataset for future analysis and machine learning tasks.

In [30]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_file = "../data/processed/flight_delays_cleaned.csv"

df_clean.to_csv(output_file, index=False)

print("Dataset Saved Successfully")
print(output_file)

Dataset Saved Successfully
../data/processed/flight_delays_cleaned.csv


The processed dataset was successfully saved.